In [1]:
import math
import random
from collections import Counter, defaultdict
from typing import List, Tuple

In [11]:
FILEPATH = "wsj_pos_tagged_en.txt"  # path to uploaded file
K = 5 
smoothing_alpha = 1.0
random_seed = 42

In [3]:
def parse_sentences(filepath: str) -> List[List[Tuple[str,str]]]:
    """
    Parse the file where tokens are of the form word_TAG.
    Sentence boundary detected by token where word == '.' and tag == '.' (._.).
    Returns list of sentences; each sentence is a list of (word, tag).
    """
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    tokens = text.split()
    sentences = []
    cur = []
    for tok in tokens:
        if "_" not in tok:
            continue
        word, tag = tok.rsplit("_", 1)
        cur.append((word, tag))
        if word == "." and tag == ".":
            sentences.append(cur)
            cur = []
    if cur:
        sentences.append(cur)
    return sentences

In [4]:
def build_counts(sentences: List[List[Tuple[str,str]]]):
    """
    Build counts for HMM:
    - tag_counts[tag]
    - emission_counts[(tag, word)]
    - transition_counts[(prev_tag, tag)] with prev_tag = "<s>" at sentence start and tag->"</s>" at end
    """
    tag_counts = Counter()
    emission_counts = Counter()
    transition_counts = Counter()
    tag_set = set()
    for sent in sentences:
        prev_tag = "<s>"
        transition_counts[(None, "<s>")] += 1  # optional count of sentence starts
        for (word, tag) in sent:
            tag_set.add(tag)
            tag_counts[tag] += 1
            emission_counts[(tag, word)] += 1
            transition_counts[(prev_tag, tag)] += 1
            prev_tag = tag
        transition_counts[(prev_tag, "</s>")] += 1
    return tag_counts, emission_counts, transition_counts, tag_set

In [5]:
def compute_probabilities(tag_counts, emission_counts, transition_counts, tag_set, alpha=1.0):
    """
    Compute add-alpha smoothed probabilities and return:
    - tags: sorted list of tags (from tag_counts keys)
    - emission_logprob: dict (tag,word) -> log P(word | tag)
    - transition_logprob: dict (prev_tag, tag) -> log P(tag | prev_tag)
    - vocab: set of words seen in training + '<UNK>'
    """
    vocab = {word for (tag, word) in emission_counts.keys()}
    vocab.add("<UNK>")
    V = len(vocab)
    emission_logprob = {}
    for tag in tag_counts:
        denom = tag_counts[tag] + alpha * V
        # known words for this tag
        for (t,w), c in emission_counts.items():
            if t == tag:
                emission_logprob[(tag, w)] = math.log((c + alpha) / denom)
        # unseen (UNK)
        emission_logprob[(tag, "<UNK>")] = math.log(alpha / denom)
    # transitions
    transition_logprob = {}
    prev_tags = set(prev for (prev, nxt) in transition_counts.keys())
    next_tags = set(tag_counts.keys()) | {"</s>"}
    for prev in prev_tags:
        total_prev = sum(c for (p,n), c in transition_counts.items() if p == prev)
        denom = total_prev + alpha * len(next_tags)
        for nxt in next_tags:
            c = transition_counts.get((prev, nxt), 0)
            transition_logprob[(prev, nxt)] = math.log((c + alpha) / denom)
    return sorted(tag_counts.keys()), emission_logprob, transition_logprob, vocab


In [6]:
def viterbi(sentence_words: List[str], tags: List[str], emission_logprob, transition_logprob, vocab) -> List[str]:
    """
    Viterbi decoding using log-probs.
    Unknown words are mapped to '<UNK>'.
    Returns list of predicted tags (same length as sentence_words).
    """
    if not sentence_words:
        return []
    obs = [w if w in vocab else "<UNK>" for w in sentence_words]
    T = len(obs)
    Vmat = {}       # Vmat[(t, tag)] = best log-prob up to time t with tag
    backptr = {}    # backptr[(t, tag)] = best previous tag

    # Initialization (t=0), use transition from "<s>"
    for tag in tags:
        trans_lp = transition_logprob.get(("<s>", tag), transition_logprob.get((None, tag), math.log(1e-12)))
        emis_lp = emission_logprob.get((tag, obs[0]), emission_logprob.get((tag, "<UNK>"), math.log(1e-12)))
        Vmat[(0, tag)] = trans_lp + emis_lp
        backptr[(0, tag)] = None

    # Recursion
    for t in range(1, T):
        for tag in tags:
            best_lp = -1e999
            best_prev = None
            emis_lp = emission_logprob.get((tag, obs[t]), emission_logprob.get((tag, "<UNK>"), math.log(1e-12)))
            for prev_tag in tags:
                prev_lp = Vmat.get((t-1, prev_tag), -1e999)
                trans_lp = transition_logprob.get((prev_tag, tag), math.log(1e-12))
                cand = prev_lp + trans_lp + emis_lp
                if cand > best_lp:
                    best_lp = cand
                    best_prev = prev_tag
            Vmat[(t, tag)] = best_lp
            backptr[(t, tag)] = best_prev

    # Termination: choose best last tag considering transition to </s>
    best_final_lp = -1e999
    best_final_tag = None
    for tag in tags:
        lp = Vmat.get((T-1, tag), -1e999) + transition_logprob.get((tag, "</s>"), math.log(1e-12))
        if lp > best_final_lp:
            best_final_lp = lp
            best_final_tag = tag

    # Backtrack
    preds = [best_final_tag]
    for t in range(T-1, 0, -1):
        prev = backptr[(t, preds[-1])]
        preds.append(prev)
    preds.reverse()
    return preds

In [7]:
def evaluate_predictions_union(golds: List[str], preds: List[str]):
    """
    Compute TP, FP, FN per tag. Use union of tags seen in gold or pred.
    Returns dict: tag -> [TP, FP, FN]
    """
    tags_union = set(golds) | set(preds)
    per_tag = {tag: [0,0,0] for tag in tags_union}
    for g, p in zip(golds, preds):
        if g == p:
            per_tag[g][0] += 1
        else:
            per_tag[p][1] += 1
            per_tag[g][2] += 1
    return per_tag

In [8]:
def precision_recall_f1_from_counts(tp, fp, fn):
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
    return prec, rec, f1


In [9]:

def cross_validate(sentences, K=5, alpha=1.0, seed=42):
    random.seed(seed)
    n = len(sentences)
    if n < K:
        raise ValueError(f"Not enough sentences ({n}) for K={K}")

    indices = list(range(n))
    random.shuffle(indices)
    folds = []
    fold_size = n // K
    for i in range(K):
        start = i*fold_size
        end = (i+1)*fold_size if i < K-1 else n
        fold_idx = indices[start:end]
        folds.append([sentences[idx] for idx in fold_idx])

    fold_metrics = []
    overall_per_tag_counts = {}

    for k in range(K):
        test_sents = folds[k]
        train_sents = [s for i in range(K) if i != k for s in folds[i]]

        tag_counts, emission_counts, transition_counts, tag_set = build_counts(train_sents)
        tags_list, emission_lp, transition_lp, vocab = compute_probabilities(tag_counts, emission_counts, transition_counts, tag_set, alpha=alpha)
        # Filter tags_list to exclude boundary tokens if present
        tags_list = sorted([t for t in tags_list if t not in ("<s>", "</s>")])

        fold_per_tag = defaultdict(lambda: [0,0,0])  # tp, fp, fn
        total_tokens = 0
        correct = 0

        for sent in test_sents:
            words = [w for (w,t) in sent]
            golds = [t for (w,t) in sent]
            preds = viterbi(words, tags_list, emission_lp, transition_lp, vocab)
            if len(preds) != len(golds):
                # fallback to most frequent tag in train
                most_freq_tag = max(tag_counts, key=tag_counts.get)
                preds = [most_freq_tag] * len(golds)
            # token accuracy
            for g,p in zip(golds, preds):
                total_tokens += 1
                if g == p:
                    correct += 1
            # per-tag counts
            per_tag_counts = evaluate_predictions_union(golds, preds)
            for tag, (tp,fp,fn) in per_tag_counts.items():
                fold_per_tag[tag][0] += tp
                fold_per_tag[tag][1] += fp
                fold_per_tag[tag][2] += fn

        # compute micro and macro metrics
        total_TP = sum(v[0] for v in fold_per_tag.values())
        total_FP = sum(v[1] for v in fold_per_tag.values())
        total_FN = sum(v[2] for v in fold_per_tag.values())
        micro_p, micro_r, micro_f1 = precision_recall_f1_from_counts(total_TP, total_FP, total_FN)

        per_tag_prfs = []
        for tag, (tp,fp,fn) in fold_per_tag.items():
            p,r,f = precision_recall_f1_from_counts(tp,fp,fn)
            per_tag_prfs.append((p,r,f))

        macro_p = sum(p for p,_,_ in per_tag_prfs)/len(per_tag_prfs) if per_tag_prfs else 0.0
        macro_r = sum(r for _,r,_ in per_tag_prfs)/len(per_tag_prfs) if per_tag_prfs else 0.0
        macro_f = sum(f for _,_,f in per_tag_prfs)/len(per_tag_prfs) if per_tag_prfs else 0.0
        accuracy = correct / total_tokens if total_tokens > 0 else 0.0

        fold_metrics.append({
            "fold": k+1,
            "accuracy": accuracy,
            "micro_precision": micro_p,
            "micro_recall": micro_r,
            "micro_f1": micro_f1,
            "macro_precision": macro_p,
            "macro_recall": macro_r,
            "macro_f1": macro_f,
            "per_tag_counts": dict(fold_per_tag),
            "n_test_tokens": total_tokens
        })

        # accumulate overall
        for tag, counts in fold_per_tag.items():
            if tag not in overall_per_tag_counts:
                overall_per_tag_counts[tag] = counts.copy()
            else:
                overall_per_tag_counts[tag][0] += counts[0]
                overall_per_tag_counts[tag][1] += counts[1]
                overall_per_tag_counts[tag][2] += counts[2]

    return fold_metrics, overall_per_tag_counts

def print_results(fold_metrics, overall_per_tag_counts, K, alpha, seed):
    print(f"\nPer-fold results (K = {K})")
    for m in fold_metrics:
        print(f"Fold {m['fold']}: accuracy={m['accuracy']:.4f}, micro-P={m['micro_precision']:.4f}, micro-R={m['micro_recall']:.4f}, micro-F1={m['micro_f1']:.4f}, macro-F1={m['macro_f1']:.4f}")

    tot_TP = sum(v[0] for v in overall_per_tag_counts.values())
    tot_FP = sum(v[1] for v in overall_per_tag_counts.values())
    tot_FN = sum(v[2] for v in overall_per_tag_counts.values())
    overall_micro_p, overall_micro_r, overall_micro_f1 = precision_recall_f1_from_counts(tot_TP, tot_FP, tot_FN)

    per_tag_prfs = []
    for tag, (tp,fp,fn) in overall_per_tag_counts.items():
        p,r,f = precision_recall_f1_from_counts(tp,fp,fn)
        per_tag_prfs.append((tag,p,r,f))
    macro_p = sum(x[1] for x in per_tag_prfs)/len(per_tag_prfs)
    macro_r = sum(x[2] for x in per_tag_prfs)/len(per_tag_prfs)
    macro_f1 = sum(x[3] for x in per_tag_prfs)/len(per_tag_prfs)

    print("\nCross-fold aggregated results:")
    print(f"Micro-avg Precision={overall_micro_p:.4f}, Recall={overall_micro_r:.4f}, F1={overall_micro_f1:.4f}")
    print(f"Macro-avg Precision={macro_p:.4f}, Recall={macro_r:.4f}, F1={macro_f1:.4f}")

    # top 10 tags by support (TP+FP+FN)
    tag_supports = [(tag, sum(overall_per_tag_counts[tag])) for tag in overall_per_tag_counts]
    tag_supports_sorted = sorted(tag_supports, key=lambda x: x[1], reverse=True)
    print("\nTop 10 tags by total support (TP+FP+FN) across folds and their P/R/F1:")
    for tag, _ in tag_supports_sorted[:10]:
        tp,fp,fn = overall_per_tag_counts[tag]
        p,r,f = precision_recall_f1_from_counts(tp,fp,fn)
        support = tp+fp+fn
        print(f"{tag:>6}  support={support:6}  P={p:.3f}  R={r:.3f}  F1={f:.3f}")

    print("\nNotes:")
    print(f"- K folds = {K}, smoothing alpha = {alpha}, random seed = {seed}")
    print("- Emission smoothing uses add-alpha with an explicit <UNK> token.")
    print("- Transition smoothing uses add-alpha over next-tags seen in training (tags + </s>).")

In [12]:
print("Parsing sentences from:", FILEPATH)
sentences = parse_sentences(FILEPATH)
print("Parsed sentences:", len(sentences))
fold_metrics, overall_counts = cross_validate(sentences, K=K, alpha=smoothing_alpha, seed=random_seed)
print_results(fold_metrics, overall_counts, K, smoothing_alpha, random_seed)

Parsing sentences from: wsj_pos_tagged_en.txt
Parsed sentences: 3828

Per-fold results (K = 5)
Fold 1: accuracy=0.8624, micro-P=0.8624, micro-R=0.8624, micro-F1=0.8624, macro-F1=0.6649
Fold 2: accuracy=0.8559, micro-P=0.8559, micro-R=0.8559, micro-F1=0.8559, macro-F1=0.6625
Fold 3: accuracy=0.8593, micro-P=0.8593, micro-R=0.8593, micro-F1=0.8593, macro-F1=0.6304
Fold 4: accuracy=0.8489, micro-P=0.8489, micro-R=0.8489, micro-F1=0.8489, macro-F1=0.6710
Fold 5: accuracy=0.8556, micro-P=0.8556, micro-R=0.8556, micro-F1=0.8556, macro-F1=0.6362

Cross-fold aggregated results:
Micro-avg Precision=0.8564, Recall=0.8564, F1=0.8564
Macro-avg Precision=0.7207, Recall=0.6016, F1=0.6296

Top 10 tags by total support (TP+FP+FN) across folds and their P/R/F1:
    NN  support= 15595  P=0.827  R=0.879  F1=0.852
    IN  support= 11527  P=0.852  R=0.977  F1=0.910
   NNP  support= 10939  P=0.841  R=0.859  F1=0.850
    DT  support= 10398  P=0.783  R=0.987  F1=0.873
    JJ  support=  7441  P=0.729  R=0.742 